Install

In [ ]:
!pip install google-cloud-aiplatform[agent_engines,evaluation] -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.1 MB/s eta 0:00:00


Authenticate

In [ ]:
from google.colab import auth
auth.authenticate_user()

Setup

In [ ]:
import vertexai
from vertexai.preview import reasoning_engines

PROJECT_ID = "genai-2222026"
LOCATION = "us-central1"
AGENT_ID = "2708361571758768128"  # Your deployed agent

vertexai.init(project=PROJECT_ID, location=LOCATION)

# Load your agent
agent = reasoning_engines.ReasoningEngine(
    f"projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/{AGENT_ID}"
)

 Test queries:

In [ ]:
# Test dataset
test_queries = [
    "How to setup Joule?",
    "What are prerequisites for BDC?",
    "How to integrate Joule with S/4HANA?",
]

# Run and collect responses
for query in test_queries:
    response = agent.query(input=query)
    print(f"Q: {query}")
    print(f"A: {response}\n")

AttributeError: 'ReasoningEngine' object has no attribute 'query'

In [ ]:
# Create a session and send message
session = agent.create_session(user_id="eval-user")
print(f"Session created: {session}")
print(dir(agent))

Session created: {'appName': '2708361571758768128', 'events': [], 'state': {}, 'id': '3472851869112467456', 'lastUpdateTime': 1775610094.893737, 'userId': 'eval-user'}
['_FutureManager__latest_future', '_FutureManager__latest_future_lock', '__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_are_futures_done', '_assert_gca_resource_is_available', '_complete_future', '_construct_sdk_resource_from_gapic', '_delete', '_delete_method', '_empty_constructor', '_exception', '_format_resource_name', '_format_resource_name_method', '_gca_resource', '_generate_display_name', '_get_and_validate_project_location', '_get_gca_resource', '_get

Query the Agent

In [ ]:
print(dir(agent.execution_api_client))

['WrappedClient', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_clients', '_default_version', '_is_temporary', '_version_map', 'api_endpoint', 'get_gapic_client_class', 'select_version']


REST API Query:

In [ ]:
import requests
from google.auth import default
from google.auth.transport.requests import Request

# Get credentials
credentials, project = default()
credentials.refresh(Request())

# Build request
url = f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/{AGENT_ID}:query"

headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

data = {
    "input": {
        "message": "How to setup Joule?"
    }
}

response = requests.post(url, headers=headers, json=data)
print(response.status_code)
print(response.json())

400
{'error': {'code': 400, 'message': 'Reasoning Engine Execution failed.\nPlease refer to our documentation (https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/troubleshooting/use) for checking logs and other troubleshooting tips.\nError Details: {"detail":"Agent Engine Error: Default method `query` not found. Available methods are: [\'get_session\', \'async_get_session\', \'async_add_session_to_memory\', \'async_delete_session\', \'list_sessions\', \'async_list_sessions\', \'create_session\', \'async_search_memory\', \'delete_session\', \'async_create_session\']."}', 'status': 'FAILED_PRECONDITION'}}


In [ ]:
import requests
from google.auth import default
from google.auth.transport.requests import Request
import json

# Get credentials
credentials, project = default()
credentials.refresh(Request())

# First create a session
session_url = f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/{AGENT_ID}:query"

headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

# Create session
session_data = {
    "input": {
        "method": "create_session",
        "user_id": "eval-test"
    }
}

response = requests.post(session_url, headers=headers, json=session_data)
print("Session:", response.json())

Session: {'error': {'code': 400, 'message': 'Reasoning Engine Execution failed.\nPlease refer to our documentation (https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/troubleshooting/use) for checking logs and other troubleshooting tips.\nError Details: {"detail":"Agent Engine Error: Default method `query` not found. Available methods are: [\'async_search_memory\', \'async_list_sessions\', \'async_delete_session\', \'async_create_session\', \'get_session\', \'create_session\', \'list_sessions\', \'delete_session\', \'async_add_session_to_memory\', \'async_get_session\']."}', 'status': 'FAILED_PRECONDITION'}}


In [ ]:
import requests
from google.auth import default
from google.auth.transport.requests import Request

# Get credentials
credentials, project = default()
credentials.refresh(Request())

headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

# Stream query endpoint (what Playground uses)
stream_url = f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/{AGENT_ID}:streamQuery"

query_data = {
    "input": {
        "message": "How to setup Joule?",
        "user_id": "eval-user-1"
    }
}

response = requests.post(stream_url, headers=headers, json=query_data, stream=True)
print(f"Status: {response.status_code}")

for line in response.iter_lines():
    if line:
        print(line.decode())

Status: 200
{"model_version": "gemini-2.0-flash", "content": {"parts": [{"function_call": {"id": "adk-cf698b39-bac2-4ea4-90bb-f2fcefe5107d", "args": {"query": "setup Joule"}, "name": "search_sap_docs"}}], "role": "model"}, "finish_reason": "STOP", "usage_metadata": {"candidates_token_count": 8, "candidates_tokens_details": [{"modality": "TEXT", "token_count": 8}], "prompt_token_count": 90, "prompt_tokens_details": [{"modality": "TEXT", "token_count": 90}], "total_token_count": 98, "traffic_type": "ON_DEMAND"}, "avg_logprobs": -0.0013609447050839663, "invocation_id": "e-1caa4e53-53b8-404e-aaa0-2832c861e5ec", "author": "sap_landscape_assistant", "actions": {"state_delta": {}, "artifact_delta": {}, "requested_auth_configs": {}, "requested_tool_confirmations": {}}, "long_running_tool_ids": [], "id": "8c23f49c-ce06-405b-9668-a7c8a9992d15", "timestamp": 1775610483.125545}
{"content": {"parts": [{"function_response": {"id": "adk-cf698b39-bac2-4ea4-90bb-f2fcefe5107d", "name": "search_sap_docs"

In [ ]:
import json

def query_agent(question):
    """Query agent and extract response"""
    query_data = {
        "input": {
            "message": question,
            "user_id": "eval-user"
        }
    }

    response = requests.post(stream_url, headers=headers, json=query_data, stream=True)

    final_answer = ""
    for line in response.iter_lines():
        if line:
            data = json.loads(line.decode())
            # Get the final text response
            if "content" in data and "parts" in data["content"]:
                for part in data["content"]["parts"]:
                    if "text" in part:
                        final_answer = part["text"]

    return final_answer

# Test
answer = query_agent("How to setup Joule?")
print(answer)

To integrate your system with Joule, navigate to your SAP BTP Formations, find the existing formation, and click "Include Systems". Ensure that you use the same SAP Cloud Identity Services used for your Unified Setup and that the prerequisites are completed. During the Joule Booster process, make sure to select the SAP Concur system. The Joule formation status should display as complete.

[Joule_Onboarding]



In [ ]:
# Test dataset
test_cases = [
    {
        "question": "How to setup Joule?",
        "expected": "BTP Formations, Joule Booster, Identity Services"
    },
    {
        "question": "What are prerequisites for BDC?",
        "expected": "DW Consumer role, scoped role, DSP Admin"
    },
    {
        "question": "How to integrate Joule with S/4HANA?",
        "expected": "Fiori Launchpad, BTP, Identity Authentication"
    },
]

# Run evaluation
results = []
for tc in test_cases:
    answer = query_agent(tc["question"])

    # Simple keyword matching (basic evaluation)
    keywords = tc["expected"].lower().split(", ")
    matches = sum(1 for kw in keywords if kw in answer.lower())
    score = matches / len(keywords)

    results.append({
        "question": tc["question"],
        "answer": answer[:200] + "...",
        "score": score
    })
    print(f"Q: {tc['question']}")
    print(f"Score: {score:.0%}")
    print("-" * 50)

# Average score
avg_score = sum(r["score"] for r in results) / len(results)
print(f"\nOverall Score: {avg_score:.0%}")

Q: How to setup Joule?
Score: 100%
--------------------------------------------------
Q: What are prerequisites for BDC?
Score: 100%
--------------------------------------------------
Q: How to integrate Joule with S/4HANA?
Score: 100%
--------------------------------------------------

Overall Score: 100%
